In [1]:
import os 
os.environ['JAX_JIT_PJIT_API_MERGE'] = '0'
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_platform_name', 'gpu')
from qiskit_dynamics.array import Array
Array.set_default_backend('jax')

/home/jiakai/.local/lib/python3.10/site-packages/qiskit_dynamics/dispatch/backends/jax.py:34: UserWarning: The functionality in the perturbation module of Qiskit Dynamics requires a JAX version <= 0.4.6, due to a bug in JAX versions > 0.4.6. For versions 0.4.4, 0.4.5, and 0.4.6, using the perturbation module functionality requires setting os.environ['JAX_JIT_PJIT_API_MERGE'] = '0' before importing JAX or Dynamics.
  warnings.warn(


# Define undriven system using scqubits, then convert to qiskit

In [8]:
from utils import *
import numpy as np
from qiskit.quantum_info import Operator
from qiskit_dynamics import Solver
import jax.numpy as jnp
from qiskit import pulse

EJ=8.9
EC=2.5
EL=0.5
g_strength = 0.3
E_osc = 3
qubit_level = 8
osc_level = 50


qbt = scqubits.Fluxonium(EJ=EJ,EC=EC,EL=EL,flux=0,cutoff=30,truncated_dim=qubit_level)
osc = scqubits.Oscillator(E_osc=E_osc,truncated_dim=osc_level)
hilbertspace = scqubits.HilbertSpace([qbt, osc])
hilbertspace.add_interaction(g_strength=g_strength,op1=qbt.n_operator,op2=osc.creation_operator,add_hc=True)
hilbertspace.generate_lookup()
product_to_dressed = generate_single_mapping(hilbertspace.hamiltonian())
# plot_specturum(qbt, osc, hilbertspace)

In [9]:



def qobj_to_Array(matrix):
    if type(matrix) == qutip.qobj.Qobj:
        matrix = matrix.full()
    # return Operator(matrix, input_dims = (matrix.shape[0],),output_dims = (matrix.shape[1],))
    return Array(matrix)

(evals,) = hilbertspace["evals"]
diag_matrix = np.diag(evals)
static_hamiltonian = 2 * np.pi * qobj_to_Array(diag_matrix)


In [10]:
leakage_dressed_state_osc_0 = product_to_dressed[(0,0)]
leakage_dressed_state_osc_1 = product_to_dressed[(0,1)]

a = 2 * np.pi * hilbertspace.op_in_dressed_eigenbasis(op=osc.annihilation_operator)


tot_dims = a.shape[0]
g0 = jnp.zeros(tot_dims).at[product_to_dressed[(0, 0)]].set(1).reshape(-1, 1)
e0 = jnp.zeros(tot_dims).at[product_to_dressed[(1, 0)]].set(1).reshape(-1, 1)
f0 = jnp.zeros(tot_dims).at[product_to_dressed[(2, 0)]].set(1).reshape(-1, 1)
h0 = jnp.zeros(tot_dims).at[product_to_dressed[(3, 0)]].set(1).reshape(-1, 1)

pn_op = jnp.array((a.dag()*a).full())
a_op = jnp.array(a.full())

# tot_time = 500
# matrix_element_driven = abs((a+a.dag()).data.toarray()[leakage_dressed_state_osc_0][leakage_dressed_state_osc_1])
# base_drive_amplitude = 1/tot_time
# base_drive_amplitude = base_drive_amplitude/matrix_element_driven


driven_operator =  qobj_to_Array(a+a.dag())

In [11]:
# from qiskit_dynamics.pulse import InstructionToSignals
# converter = InstructionToSignals(signal_sample_dt, carriers={"d0": carrier_freq})
# signals = converter.get_signals(square)
# fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
# for ax, title in zip(axs, ["envelope", "signal"]):
#     signals[0].draw(0, tot_time/2000, 2000, title, axis=ax)
#     ax.set_xlabel("Time (ns)")
#     ax.set_ylabel("Amplitude")
#     ax.set_title(title)


In [12]:
tot_time = 10
tlist =  jnp.linspace(0,tot_time, int(tot_time))
w_d = transition_frequency(hilbertspace,leakage_dressed_state_osc_0,leakage_dressed_state_osc_1 )
carrier_freq = w_d * 2 * np.pi

signal_sample_dt = 0.01 # Sample rate of the backend in ns.

base_drive_amplitude = 0.03 * 2 * np.pi

with pulse.build(name="square") as square:
    pulse.play(pulse.Constant(duration = int(tot_time/signal_sample_dt), amp = base_drive_amplitude), pulse.DriveChannel(0))

# square.draw()

kappa = 0.5*np.pi/75
L_ops = [np.sqrt(kappa)*qobj_to_Array(a)/10]


ham_solver =  Solver(
                hamiltonian_operators=[driven_operator],
                static_hamiltonian=static_hamiltonian,
                static_dissipators=L_ops,
                # rotating_frame=static_hamiltonian,
                hamiltonian_channels=['d0'],
                channel_carrier_freqs={'d0': carrier_freq},
                dt=signal_sample_dt
            )


# Basically it's not possible to evolve a dense high-dimensional superket.

In [14]:
result =solve_with_jax_gpu_lindbladian(ham_solver = ham_solver, y0 = g0, tlist = tlist, signals = square, max_dt=1, chunk_size=0.001)

2023-08-28 22:54:57.940824: W external/tsl/tsl/framework/bfc_allocator.cc:485] Allocator (GPU_0_bfc) ran out of memory trying to allocate 381.47GiB (rounded to 409600000000)requested by op 
2023-08-28 22:54:57.941005: W external/tsl/tsl/framework/bfc_allocator.cc:497] ******************************************************************__________________________________
2023-08-28 22:54:57.941071: E external/xla/xla/pjrt/pjrt_stream_executor_client.cc:2593] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 409600000000 bytes.
BufferAssignment OOM Debugging.
BufferAssignment stats:
             parameter allocation:    3.66MiB
              constant allocation:         0B
        maybe_live_out allocation:  381.47GiB
     preallocated temp allocation:         0B
                 total allocation:  381.47GiB
              total fragmentation:         0B (0.00%)
Peak buffers:
	Buffer 1:
		Size: 381.47GiB
		Operator: op_name="jit(kron)/jit(main)/reshape

XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 409600000000 bytes.
BufferAssignment OOM Debugging.
BufferAssignment stats:
             parameter allocation:    3.66MiB
              constant allocation:         0B
        maybe_live_out allocation:  381.47GiB
     preallocated temp allocation:         0B
                 total allocation:  381.47GiB
              total fragmentation:         0B (0.00%)
Peak buffers:
	Buffer 1:
		Size: 381.47GiB
		Operator: op_name="jit(kron)/jit(main)/reshape[new_sizes=(160000, 160000) dimensions=None]" source_file="/home/jiakai/.local/lib/python3.10/site-packages/qiskit_dynamics/array/array.py" source_line=290
		XLA Label: fusion
		Shape: c128[160000,160000]
		==========================

	Buffer 2:
		Size: 2.44MiB
		Entry Parameter Subshape: c128[400,400]
		==========================

	Buffer 3:
		Size: 1.22MiB
		Entry Parameter Subshape: f64[400,400]
		==========================



In [8]:

results = []
for psi0 in [g0,e0,f0,h0]:
    result =solve_with_jax_gpu_lindbladian(ham_solver=ham_solver,
                    y0=psi0,
                    tlist=tlist,
                    signals=square,
                    chunk_size=0.01
                )
    results.append(result)

TypeError: iteration over a 0-d array

In [ ]:
plot_population(results,qubit_level,osc_level,product_to_dressed,a,w_d,tlist,fourier=True)

TypeError: dot_general requires contracting dimensions to have the same shape, got (16,) and (256,).